# llama

> The same `Chat` API as `rishi.core`, over [llama-cpp-python](https://github.com/abetlen/llama-cpp-python) GGUF models - OpenAI-style message helpers, tool schemas via [toolslm](https://github.com/AnswerDotAI/toolslm), a Python-side tool loop with human-in-the-loop approval, `<think>` channel handling, sync/async streaming, and usage tracking.

In [ ]:
#| default_exp llama

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, re, os, uuid, asyncio
from base64 import b64encode
from llama_cpp import Llama
from toolslm.funccall import get_schema, mk_ns, call_func
from huggingface_hub import hf_hub_download, list_repo_files, scan_cache_dir
from fastcore.all import Path, store_attr, patch, L, GetAttr, ifnone, detect_mime, first, listify, AttrDict
from rishi import core
from rishi.core import (UsageStats, ChatCallback, run_cbs, resp_text, thought, Resp, StreamFormatter,
                        display_stream, mk_tr_details, truncated, hitl_policy, extract_fence,
                        _matches, _qa_sp, _tool_reminder)

In [ ]:
#| export
_all_ = ['UsageStats', 'ChatCallback', 'run_cbs', 'resp_text', 'thought', 'Resp', 'StreamFormatter',
         'display_stream', 'mk_tr_details', 'truncated', 'hitl_policy', 'extract_fence']

## Messages

llama.cpp speaks the OpenAI chat schema, so messages are plain dicts. These helpers mirror `rishi.core.mk_msg`/`mk_msgs` but build OpenAI-style messages instead of litert `Message`s.

- `mk_content` maps one value to an OpenAI content part: a `str` becomes text; `bytes` or a `Path` become a base64 `image_url` part (images need a multimodal chat handler, e.g. a Llava-style model with its clip projector).
- `mk_msg` wraps content into a message dict (default role `user`); a dict passes through. All-text parts collapse to a plain string for maximum template compatibility.
- `mk_msgs` normalises a mixed list, used to seed `Chat(messages=...)`.

In [ ]:
#| export
def mk_content(o):
    'Convert `o` to an OpenAI-style content part (text, or a base64 `image_url` for image bytes/files).'
    if isinstance(o, dict): return o
    if isinstance(o, str): return {'type': 'text', 'text': o}
    if isinstance(o, os.PathLike): o = Path(o).read_bytes()
    if isinstance(o, bytes):
        mime = detect_mime(o) or 'application/octet-stream'
        if not mime.startswith('image/'): raise TypeError(f"llama.cpp chat supports only image attachments, got {mime}")
        return {'type': 'image_url', 'image_url': {'url': f"data:{mime};base64,{b64encode(o).decode()}"}}
    raise TypeError(f"Unsupported content type: {type(o)}")

def mk_msg(content, role='user'):
    'Create an OpenAI-style message dict from str/bytes/list/dict.'
    if content is None or isinstance(content, dict): return content
    parts = [mk_content(o) for o in content] if isinstance(content, list) else [mk_content(content)]
    if all(p.get('type') == 'text' for p in parts): return {'role': role, 'content': '\n'.join(p['text'] for p in parts)}
    return {'role': role, 'content': parts}

def mk_msgs(msgs):
    'Normalize a list of messages to OpenAI-style dicts.'
    return [mk_msg(m) for m in listify(msgs)] if msgs else []

In [ ]:
from fastcore.test import test_eq, test_fail

In [ ]:
test_eq(mk_msg("hello"), {'role': 'user', 'content': 'hello'})
test_eq(mk_msg(["a", "b"]), {'role': 'user', 'content': 'a\nb'})
test_eq(mk_msg({'role': 'assistant', 'content': 'hi'}), {'role': 'assistant', 'content': 'hi'})
test_eq(mk_msgs(['hi', {'role': 'assistant', 'content': 'hello'}]),
        [{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'hello'}])
png = b'\x89PNG\r\n\x1a\n' + b'0' * 8
assert mk_msg([png, 'what is this?'])['content'][0]['type'] == 'image_url'
test_fail(lambda: mk_content(b'RIFF0000WAVE'), contains='image')

## Tool schemas

`mk_tool` turns a python callable into an OpenAI/llama.cpp tool spec using toolslm's `get_schema` (docstrings and [docments](https://fastcore.fast.ai/docments.html)-style comments become descriptions). Spec dicts pass through, so you can hand-write schemas too.

In [ ]:
#| export
def mk_tool(f):
    'OpenAI/llama.cpp tool spec for callable `f` (via toolslm `get_schema`); spec dicts pass through.'
    if isinstance(f, dict): return f
    return {'type': 'function', 'function': get_schema(f, pname='parameters')}

In [ ]:
def _add(
    a:int, # first addend
    b:int=0 # second addend
):
    'Add two integers.'
    return a + b

s = mk_tool(_add)
test_eq(s['type'], 'function')
test_eq(s['function']['name'], '_add')
test_eq(s['function']['parameters']['required'], ['a'])
test_eq(mk_tool(s), s)

## Think tags and tool-call tags

Reasoning GGUF models (Qwen3, DeepSeek-R1 distills, ...) put their thinking inside `<think>...</think>`, and Hermes-format models emit tool calls as `<tool_call>{json}</tool_call>` text. llama.cpp's generic chat handler passes both through verbatim, so we parse them ourselves:

- `split_think` moves think blocks into a separate thought string (which `Chat` reports via `channels.thought` and keeps out of the context on later turns, like litert's `filter_think`).
- `parse_tool_tags` extracts tool-call blocks into OpenAI-style `tool_calls` entries.

In [ ]:
#| export
_think_re = re.compile(r'<think>(.*?)</think>', re.DOTALL)
_toolcall_re = re.compile(r'<tool_call>\s*(.*?)\s*</tool_call>', re.DOTALL)

def split_think(text):
    "Split `<think>...</think>` blocks out of `text`; returns `(clean_text, thought)`."
    text = text or ''
    ths = [m.strip() for m in _think_re.findall(text)]
    text = _think_re.sub('', text)
    if '<think>' in text:  # unterminated, e.g. cut off at the token cap
        text, _, rest = text.partition('<think>')
        ths.append(rest.strip())
    return text.strip('\n'), '\n'.join(th for th in ths if th)

def _mk_tag_tc(s):
    "Build a tool_call dict from the JSON inside a `<tool_call>` block, or None."
    try: d = json.loads(s)
    except (json.JSONDecodeError, TypeError): return None
    if not isinstance(d, dict) or 'name' not in d: return None
    return {'id': f"call_{uuid.uuid4().hex[:8]}", 'type': 'function',
            'function': {'name': d.get('name', ''), 'arguments': d.get('arguments') or {}}}

def parse_tool_tags(text):
    "Parse Hermes/Qwen-style `<tool_call>{json}</tool_call>` blocks; returns `(clean_text, tool_calls)`."
    tcs = [tc for m in _toolcall_re.findall(text or '') if (tc := _mk_tag_tc(m))]
    return _toolcall_re.sub('', text or '').strip('\n'), tcs

In [ ]:
test_eq(split_think('<think>hmm</think>\n\nhi'), ('hi', 'hmm'))
test_eq(split_think('no tags'), ('no tags', ''))
test_eq(split_think('<think>cut off mid-'), ('', 'cut off mid-'))
txt, tcs = parse_tool_tags('calling now\n<tool_call>{"name": "add", "arguments": {"a": 1}}</tool_call>')
test_eq(txt, 'calling now')
test_eq(tcs[0]['function'], {'name': 'add', 'arguments': {'a': 1}})
test_eq(parse_tool_tags('<tool_call>not json</tool_call>')[1], [])

## Response normalization

`norm_resp` converts a llama.cpp chat completion into the same shape `rishi.core` responses use - an assistant dict with `content`, `channels.thought`, parsed `tool_calls` (JSON-string arguments become dicts; `<tool_call>` tags are folded in), a `truncated` flag on `finish_reason == 'length'`, and the `usage` block - wrapped in `Resp` so it renders as markdown in notebooks.

`_oai_msg` is the reverse projection used when re-sending history: rishi-only keys (`channels`, `usage`, ...) are dropped - so thinking never re-enters the context - and tool-call arguments are re-encoded as JSON strings.

In [ ]:
#| export
def _parse_args(a):
    "Parse OpenAI JSON-string tool arguments to a dict (dicts pass through)."
    if isinstance(a, dict): return a
    try: return json.loads(a) if a else {}
    except json.JSONDecodeError: return {}

def norm_resp(r):
    "Normalize a llama.cpp chat completion to a rishi-style `Resp` dict."
    ch = r['choices'][0]
    m = ch.get('message') or {}
    text, th = split_think(m.get('content') or '')
    text, tag_tcs = parse_tool_tags(text)
    tcs = [{'id': tc.get('id'), 'type': 'function',
            'function': {'name': tc.get('function', {}).get('name', ''),
                         'arguments': _parse_args(tc.get('function', {}).get('arguments'))}}
           for tc in (m.get('tool_calls') or [])] + tag_tcs
    res = {'role': 'assistant', 'content': text}
    if th: res['channels'] = {'thought': th}
    if tcs: res['tool_calls'] = tcs
    if ch.get('finish_reason') == 'length': res['truncated'] = True
    if 'usage' in r: res['usage'] = dict(r['usage'])
    return Resp(res)

def _oai_msg(m):
    "Project a history entry to what llama.cpp expects: drop rishi-only keys, re-encode tool-call args as JSON."
    out = {k: m[k] for k in ('role', 'content', 'tool_calls', 'tool_call_id', 'name') if m.get(k) is not None}
    if 'tool_calls' in out:
        out['tool_calls'] = [{'id': tc.get('id') or f'call_{i}', 'type': 'function',
                              'function': {'name': tc['function'].get('name', ''),
                                           'arguments': a if isinstance((a := tc['function'].get('arguments')), str)
                                                        else json.dumps(a or {})}}
                             for i, tc in enumerate(out['tool_calls'])]
        out.setdefault('content', '')
    return out

def _sum_usage(us):
    "Sum OpenAI usage dicts (ignoring Nones); None if nothing to sum."
    us = [u for u in us if u]
    if not us: return None
    return {k: sum(u.get(k, 0) for u in us) for k in ('prompt_tokens', 'completion_tokens', 'total_tokens')}

In [ ]:
raw = {'choices': [{'message': {'role': 'assistant',
                               'content': '<think>2+3</think>The answer is 5.\n<tool_call>{"name": "add", "arguments": {"a": 2, "b": 3}}</tool_call>',
                               'tool_calls': [{'id': 'x1', 'type': 'function',
                                               'function': {'name': 'mul', 'arguments': '{"a": 4}'}}]},
                    'finish_reason': 'stop'}],
       'usage': {'prompt_tokens': 10, 'completion_tokens': 5, 'total_tokens': 15}}
r = norm_resp(raw)
test_eq(resp_text(r), 'The answer is 5.')
test_eq(thought(r), '2+3')
test_eq([tc['function']['name'] for tc in r['tool_calls']], ['mul', 'add'])
test_eq(r['tool_calls'][0]['function']['arguments'], {'a': 4})
assert not truncated(r)

o = _oai_msg(r)
assert 'channels' not in o and 'usage' not in o
test_eq(o['tool_calls'][0]['function']['arguments'], '{"a": 4}')
test_eq(_oai_msg({'role': 'tool', 'tool_call_id': 't1', 'name': 'add', 'content': '5'}),
        {'role': 'tool', 'tool_call_id': 't1', 'name': 'add', 'content': '5'})
test_eq(_sum_usage([{'prompt_tokens': 1, 'completion_tokens': 2, 'total_tokens': 3}, None,
                    {'prompt_tokens': 10, 'completion_tokens': 20, 'total_tokens': 30}]),
        {'prompt_tokens': 11, 'completion_tokens': 22, 'total_tokens': 33})

## Streaming

`StreamSplit` is a stateful splitter for streamed text: it routes `<think>...</think>` content to the thought channel and holds back `<tool_call>` blocks (so raw JSON never hits the display), emitting litert-style chunk dicts that `rishi.core.StreamFormatter` already knows how to render. Tags split across chunk boundaries are handled by holding back any suffix that could still become a tag. `_acc_tc` accumulates structured `tool_calls` deltas (from chat handlers with native tool support).

In [ ]:
#| export
_tags = ('<think>', '</think>', '<tool_call>', '</tool_call>')

class StreamSplit:
    "Stateful splitter for streamed text: `<think>` -> thought chunks, `<tool_call>` blocks held back and parsed."
    def __init__(self):
        self.buf, self.state, self.text, self.thought, self.tool_calls, self._tc_buf, self._strip = '', 'text', '', '', [], '', False
    def _held(self):
        "Length of the longest `buf` suffix that could still become a tag."
        for n in range(min(len(self.buf), max(map(len, _tags)) - 1), 0, -1):
            if any(t.startswith(self.buf[-n:]) for t in _tags): return n
        return 0
    def _emit_text(self, out):
        if self._strip: out = out.lstrip('\n')
        if not out: return None
        self._strip = False; self.text += out
        return {'content': [{'type': 'text', 'text': out}]}
    def feed(self, s):
        "Consume a text delta; yield litert-style chunk dicts."
        self.buf += s
        while True:
            if self.state == 'text':
                cands = [(k, t, st) for k, t, st in ((self.buf.find('<think>'), '<think>', 'think'),
                                                     (self.buf.find('<tool_call>'), '<tool_call>', 'tool')) if k >= 0]
                if not cands:
                    n = self._held()
                    out, self.buf = self.buf[:len(self.buf) - n], self.buf[len(self.buf) - n:]
                    if (c := self._emit_text(out)): yield c
                    return
                k, tag, st = min(cands)
                out, self.buf, self.state = self.buf[:k], self.buf[k + len(tag):], st
                if (c := self._emit_text(out)): yield c
            elif self.state == 'think':
                k = self.buf.find('</think>')
                if k < 0:
                    n = self._held()
                    out, self.buf = self.buf[:len(self.buf) - n], self.buf[len(self.buf) - n:]
                    if out: self.thought += out; yield {'channels': {'thought': out}}
                    return
                out, self.buf, self.state, self._strip = self.buf[:k], self.buf[k + len('</think>'):], 'text', True
                if out: self.thought += out; yield {'channels': {'thought': out}}
            else:  # tool
                k = self.buf.find('</tool_call>')
                if k < 0:
                    n = self._held()
                    self._tc_buf += self.buf[:len(self.buf) - n]; self.buf = self.buf[len(self.buf) - n:]
                    return
                self._tc_buf += self.buf[:k]
                self.buf, self.state, self._strip = self.buf[k + len('</tool_call>'):], 'text', True
                if (tc := _mk_tag_tc(self._tc_buf)): self.tool_calls.append(tc)
                self._tc_buf = ''
    def finish(self):
        "Flush leftovers: unterminated think becomes thought; an unclosed tool block is parsed if possible."
        s, self.buf = self.buf, ''
        if self.state == 'think':
            if s: self.thought += s; yield {'channels': {'thought': s}}
        elif self.state == 'tool':
            if (tc := _mk_tag_tc(self._tc_buf + s)): self.tool_calls.append(tc)
            self._tc_buf = ''
        elif (c := self._emit_text(s)): yield c

def _acc_tc(acc, deltas):
    "Fold streamed OpenAI `tool_calls` deltas into `acc` (a list of partial tool_call dicts)."
    for d in deltas or []:
        i = d.get('index', 0)
        while len(acc) <= i: acc.append({'id': None, 'type': 'function', 'function': {'name': '', 'arguments': ''}})
        if d.get('id'): acc[i]['id'] = d['id']
        f = d.get('function') or {}
        if f.get('name'): acc[i]['function']['name'] += f['name']
        if f.get('arguments'): acc[i]['function']['arguments'] += f['arguments']

In [ ]:
def _run_split(deltas):
    sp, chunks = StreamSplit(), []
    for d in deltas: chunks += list(sp.feed(d))
    chunks += list(sp.finish())
    return sp, chunks

# tags split across chunk boundaries
sp, chunks = _run_split(['<thi', 'nk>let me', ' see</th', 'ink>\n\nans', 'wer: 5 <tool', '_call>{"name": "add", "arguments": {"a": 1}}</tool_call>'])
test_eq(sp.thought, 'let me see')
test_eq(sp.text, 'answer: 5 ')
test_eq(sp.tool_calls[0]['function'], {'name': 'add', 'arguments': {'a': 1}})
assert all(('channels' in c) or ('content' in c) for c in chunks)

# plain text passes through; unterminated think flushes to thought
sp, _ = _run_split(['just ', 'text'])
test_eq((sp.text, sp.thought, sp.tool_calls), ('just text', '', []))
sp, _ = _run_split(['<think>never closed'])
test_eq(sp.thought, 'never closed')

# a lone '<' that never becomes a tag is still emitted
sp, _ = _run_split(['a < b', ' and c'])
test_eq(sp.text, 'a < b and c')

acc = []
_acc_tc(acc, [{'index': 0, 'id': 'c1', 'function': {'name': 'add', 'arguments': ''}}])
_acc_tc(acc, [{'index': 0, 'function': {'arguments': '{"a"'}}])
_acc_tc(acc, [{'index': 0, 'function': {'arguments': ': 2}'}}])
test_eq(acc, [{'id': 'c1', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": 2}'}}])

## Built-in callbacks

The callback system is shared with `rishi.core` (`ChatCallback`, `run_cbs`, and the same event names), so callbacks written for one backend generally work on the other. llama.cpp is stateless per call, though, so here `chat.hist` *is* the conversation state and `Chat` maintains it directly - there is no `HistoryCallback`. `UsageCallback` folds the turn's `usage` block (summed across tool rounds) into `chat.use`, and `ToolReminderCallback` injects the same reminder text as the litert backend.

In [ ]:
#| export
class UsageCallback(ChatCallback):
    "Fold each turn's `usage` block (summed across tool rounds) into `chat.use`."
    order = 10
    def after_response(self):
        u = self.chat.turn_res.get('usage') or {}
        self.chat.use += UsageStats(u.get('prompt_tokens', 0), u.get('completion_tokens', 0), u.get('total_tokens', 0), 1)

class ToolReminderCallback(ChatCallback):
    'Inject a tool-summary reminder into the outgoing message when tools are registered.'
    order = 30
    def __init__(self, tool_reminder=_tool_reminder): store_attr()
    def before_send(self):
        m = self.chat.turn_msg
        if not self.chat.tools or m is None: return
        if isinstance(m.get('content'), str): m['content'] += self.tool_reminder
        elif isinstance(m.get('content'), list): m['content'].append({'type': 'text', 'text': self.tool_reminder})

## Loading GGUF models

Any GGUF repo on the HuggingFace Hub works; the Qwen3 and Gemma 3 ids below are handy defaults (Qwen3 gives you thinking *and* Hermes-style tool calls out of the box). `get_model` resolves a `.gguf` file with the same cache-first ladder as `rishi.core.get_model`: explicit `model_path`, then the local HF cache (no network), then a download - preferring the requested `quant` and skipping `mmproj` projector files.

In [ ]:
#| export
qwen3_06b = 'Qwen/Qwen3-0.6B-GGUF'
qwen3_17b = 'Qwen/Qwen3-1.7B-GGUF'
qwen3_4b  = 'Qwen/Qwen3-4B-GGUF'
gemma3_1b = 'ggml-org/gemma-3-1b-it-GGUF'
gemma3_4b = 'ggml-org/gemma-3-4b-it-GGUF'

def _gguf(fs, quant='Q4_K_M'):
    "First `.gguf` path in `fs` matching `quant` (case-insensitive), else the first one (skips `mmproj` files)."
    ggufs = L(fs).filter(lambda p: p.lower().endswith('.gguf') and 'mmproj' not in p.lower())
    return first(ggufs, lambda p: quant.lower() in p.lower()) or first(ggufs)

def _cached_model(model_id, quant='Q4_K_M'):
    "Local `.gguf` path from the HF cache without hitting the network, else None."
    try: repo = first(scan_cache_dir().repos, lambda r: r.repo_id == model_id)
    except Exception: return None
    return _gguf([str(f.file_path) for r in repo.revisions for f in r.files], quant) if repo else None

def get_model(model_id, model_path=None, quant='Q4_K_M'):
    "Return a local `.gguf` path: `model_path`, else HF cache, else download."
    if model_path and Path(model_path).exists(): return str(model_path)
    if (hit := _cached_model(model_id, quant)): return hit
    if not (fn := _gguf(list_repo_files(model_id), quant)): raise FileNotFoundError(f"No .gguf file found for {model_id}")
    return hf_hub_download(model_id, fn)

In [ ]:
fs = ['README.md', 'model-Q8_0.gguf', 'model-Q4_K_M.gguf', 'mmproj-model.gguf']
test_eq(_gguf(fs), 'model-Q4_K_M.gguf')
test_eq(_gguf(fs, 'Q8_0'), 'model-Q8_0.gguf')
test_eq(_gguf(fs, 'IQ2_XS'), 'model-Q8_0.gguf')  # falls back to first non-mmproj gguf
test_eq(_gguf(['README.md']), None)

## Chat

Same shape as `rishi.core.Chat`: build it (it constructs or accepts a `llama_cpp.Llama`), then call it like a function - one turn per call, updating history and usage; `stream=True` yields markdown chunks instead. `create_engine` is a patchable classmethod; pass a prebuilt `engine=` to share one model across chats.

The tool loop runs in Python (litert runs it inside the engine): tools are passed to the chat template *and* described to llama.cpp, responses are checked for structured `tool_calls` or `<tool_call>` tags, each call is routed through `approve` (so `hitl_policy` works unchanged), executed via toolslm's `call_func`, and fed back as `role='tool'` messages - up to `max_steps` rounds per turn.

`think` uses the Qwen-style soft switch: `True`/`False` appends `/think` or `/no_think` to the system prompt; `None` leaves the model's default. Thinking is split out of replies into `channels.thought` and never re-sent, mirroring litert's `filter_think`.

In [ ]:
#| export
_dflt_cbs = [UsageCallback, ToolReminderCallback]

class Chat:
    "Sync chat over a local llama.cpp model - the `rishi.core.Chat` API with a Python-side tool loop."
    @classmethod
    def create_engine(cls, model_id=qwen3_17b, model_path=None, quant='Q4_K_M', n_ctx=8192,
                      n_gpu_layers=0, verbose=False, **kw):
        'Build a `llama_cpp.Llama`. Override/`@patch` to customize.'
        return Llama(get_model(model_id, model_path, quant), n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, verbose=verbose, **kw)

    def __init__(self,
        engine:Llama=None, # llama_cpp.Llama, or None to build one
        model_id:str=qwen3_17b, # huggingface GGUF repo id
        model_path:os.PathLike=None, # local .gguf model path
        quant:str='Q4_K_M', # preferred quant when picking a .gguf from the repo
        n_ctx:int=8192, # context window for the engine
        n_gpu_layers:int=0, # layers to offload to GPU (-1 = all)
        eng_kw=None, # extra llama_cpp.Llama kwargs
        sp='', # system prompt
        messages=None, # message history to prefill the conversation
        tools=None, # tools: python callables or OpenAI tool-spec dicts
        ctx_limit=None, # context window for pct_full (defaults to the engine's n_ctx)
        approve=None, # approval function for tool calls
        tool_max_len=None, # truncate string tool results longer than this (protects context)
        max_steps=10, # max tool-call rounds per turn
        think=None, # True/False appends '/think' or '/no_think' to the system prompt (Qwen-style soft switch)
        temp=None, top_k=None, top_p=None, seed=None, # sampler knobs
        max_output_tokens=None,
        comp_kw=None, # extra create_chat_completion kwargs
        cbs=None, # callbacks to register
        default_cbs=True # add default callbacks (UsageCallback, ToolReminderCallback)
    ):
        self._own_engine = engine is None
        if engine is None:
            engine = self.create_engine(model_id, model_path, quant, n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, **(eng_kw or {}))
        self.engine = engine
        self.tools = L(tools)
        self.toolspecs = [mk_tool(t) for t in self.tools]
        self.ns = mk_ns([t for t in self.tools if callable(t)])
        self.hist = mk_msgs(messages)
        self._samp = {k: v for k, v in dict(temperature=temp, top_k=top_k, top_p=top_p, seed=seed).items() if v is not None}
        store_attr('sp,approve,tool_max_len,max_steps,think,max_output_tokens')
        self.ctx_limit, self.comp_kw = ifnone(ctx_limit, engine.n_ctx()), comp_kw or {}
        self.use, self.cbs, self.turn_msg, self.turn_res, self._ctx_tokens = UsageStats(), L(), None, None, 0
        if default_cbs: self.add_cbs(_dflt_cbs)
        self.add_cbs(cbs)

    add_cb, add_cbs, remove_cb, remove_cbs, print_hist = (
        core.Chat.add_cb, core.Chat.add_cbs, core.Chat.remove_cb, core.Chat.remove_cbs, core.Chat.print_hist)

    def close(self):
        "Release the engine (only if this Chat created it); idempotent."
        if self._own_engine and getattr(self, 'engine', None) is not None:
            if (c := getattr(self.engine, 'close', None)): c()
            self.engine = None
    def __del__(self):
        try: self.close()
        except Exception: pass

    @property
    def token_count(self):
        "Tokens in the model context after the last turn (prompt + completion)."
        return self._ctx_tokens
    @property
    def pct_full(self): return self.token_count / self.ctx_limit
    def count_tokens(self, text):
        "Number of tokens in `text` per the engine tokenizer."
        return len(self.engine.tokenize(text.encode('utf-8'), add_bos=False, special=True))

    def _sys_msgs(self):
        sp = self.sp or ''
        if self.think is not None: sp = f"{sp}\n{'/think' if self.think else '/no_think'}".strip()
        return [{'role': 'system', 'content': sp}] if sp else []

    def _msgs(self): return self._sys_msgs() + [_oai_msg(m) for m in self.hist]

    def _step(self, max_output_tokens=None, stream=False):
        return self.engine.create_chat_completion(self._msgs(), tools=self.toolspecs or None, stream=stream,
            max_tokens=ifnone(max_output_tokens, self.max_output_tokens), **self._samp, **self.comp_kw)

    def _run_tools(self, res):
        "Record `res` in `hist`, then execute its tool calls (with approval), appending the results."
        self.hist.append(res)
        for tc in res.get('tool_calls') or []:
            self.turn_tc = tc
            for _ in run_cbs(self, 'before_tool_calls'): pass
            if self.approve is None or self.approve(tc):
                try: out = call_func(tc['function']['name'], tc['function']['arguments'], ns=self.ns)
                except Exception as e: out = f"{type(e).__name__}: {e}"
            else: out = 'Denied by human operator'
            if self.tool_max_len and isinstance(out, str) and len(out) > self.tool_max_len:
                out = out[:self.tool_max_len] + ' ...[truncated]'
            self.turn_tool_result = out
            self.hist.append({'role': 'tool', 'tool_call_id': tc.get('id'), 'name': tc['function'].get('name', ''),
                              'content': str(out)})
            for _ in run_cbs(self, 'after_tool_calls'): pass

    def _send(self, msg, max_output_tokens=None):
        'Send one message through the callback pipeline, running the tool loop up to `max_steps` rounds.'
        self.turn_msg = mk_msg(msg)
        for _ in run_cbs(self, 'before_send'): pass
        if self.turn_msg is not None: self.hist.append(self.turn_msg)
        us, steps = [], 0
        while True:
            res = norm_resp(self._step(max_output_tokens))
            us.append(res.get('usage'))
            if not res.get('tool_calls') or steps >= self.max_steps: break
            self._run_tools(res); steps += 1
        if (u := _sum_usage(us)): res['usage'] = u
        self._ctx_tokens = (us[-1] or {}).get('total_tokens', self._ctx_tokens)
        self.turn_res = res
        self.hist.append(res)
        for _ in run_cbs(self, 'after_response'): pass
        return self.turn_res

    def _stream_step(self, max_output_tokens=None):
        "Stream one completion; yields chunk dicts and leaves the merged `Resp` on `self._step_res`."
        split, tcs, fin = StreamSplit(), [], None
        for r in self._step(max_output_tokens, stream=True):
            ch = r['choices'][0]
            fin = ch.get('finish_reason') or fin
            d = ch.get('delta') or {}
            _acc_tc(tcs, d.get('tool_calls'))
            yield from split.feed(d.get('content') or '')
        yield from split.finish()
        tcs = [{'id': tc['id'] or f'call_{i}', 'type': 'function',
                'function': {'name': tc['function']['name'], 'arguments': _parse_args(tc['function']['arguments'])}}
               for i, tc in enumerate(tcs)] + split.tool_calls
        res = {'role': 'assistant', 'content': split.text}
        if split.thought: res['channels'] = {'thought': split.thought}
        if tcs: res['tool_calls'] = tcs
        if fin == 'length': res['truncated'] = True
        out = self.count_tokens(split.text + split.thought) if (split.text or split.thought) else 0
        n = self.engine.n_tokens
        res['usage'] = {'prompt_tokens': max(n - out, 0), 'completion_tokens': out, 'total_tokens': n}
        self._step_res = Resp(res)

    def _stream(self, msg, max_output_tokens=None, cbs=None):
        'Stream a turn as markdown chunks; per-call `cbs` live only for this turn.'
        added = self.add_cbs(cbs)
        try:
            self.turn_msg = mk_msg(msg)
            for _ in run_cbs(self, 'before_send'): pass
            if self.turn_msg is not None: self.hist.append(self.turn_msg)
            fmt, us, steps = StreamFormatter(), [], 0
            while True:
                for o in self._stream_step(max_output_tokens):
                    if (s := fmt.format_item(o)): yield s
                if fmt._inthink: fmt._inthink = False; yield '\n\n'
                res = self._step_res
                us.append(res.get('usage'))
                if not res.get('tool_calls') or steps >= self.max_steps: break
                for tc in res['tool_calls']:
                    yield fmt.format_item({'content': [{'type': 'tool_call', 'name': tc['function'].get('name', ''),
                                                        'arguments': tc['function'].get('arguments', {})}]})
                self._run_tools(res); steps += 1
            if (u := _sum_usage(us)): res['usage'] = u
            self._ctx_tokens = (us[-1] or {}).get('total_tokens', self._ctx_tokens)
            self.turn_res = res
            self.hist.append(res)
            for _ in run_cbs(self, 'after_response'): pass
        finally: self.remove_cbs(added)

    def __call__(self, msg:list|str|dict|bytes|os.PathLike=None, stream=False, max_output_tokens=None,
                 cbs=None # extra callbacks for this turn only (added before, removed after)
    ):
        'Run one chat turn; returns a `Resp` dict (or a markdown-chunk generator when `stream=True`).'
        self.use = UsageStats()
        if stream: return self._stream(msg, max_output_tokens, cbs)
        added = self.add_cbs(cbs)
        try: return self._send(msg, max_output_tokens)
        finally: self.remove_cbs(added)

## Utilities: classify, structured, run_py

Same one-shot helpers as the litert backend, run stateless on this chat's engine (isolated from the conversation). `structured` forces the tool call with `tool_choice`, which llama.cpp turns into a JSON-schema grammar - so the arguments always parse. `run_py` and `grades` are shared with `rishi.core` directly, so `PyFenceCallback` works on this backend too.

In [ ]:
#| export
@patch
def classify(self:Chat, text, labels, sp='Reply with only the single best label and nothing else.'):
    "One-shot label for `text`, run stateless on this chat's engine (isolated from the conversation)."
    p = f"{text}\n\nChoose exactly one label from: {', '.join(labels)}."
    msgs = ([{'role': 'system', 'content': sp}] if sp else []) + [{'role': 'user', 'content': p}]
    out = resp_text(norm_resp(self.engine.create_chat_completion(msgs, **self._samp))).lower()
    return first(labels, lambda l: l.lower() in out) or out.strip()

@patch
def structured(self:Chat, prompt, schema, sp='Call the tool to answer.'):
    "One-shot structured output via a forced (grammar-constrained) tool call; returns `schema(**arguments)`."
    spec = mk_tool(schema)
    nm = spec['function']['name']
    msgs = ([{'role': 'system', 'content': sp}] if sp else []) + [{'role': 'user', 'content': prompt}]
    r = norm_resp(self.engine.create_chat_completion(msgs, tools=[spec],
            tool_choice={'type': 'function', 'function': {'name': nm}}, **self._samp))
    if not (tcs := r.get('tool_calls')): raise ValueError(f"model did not call the tool; reply: {resp_text(r)[:200]!r}")
    return schema(**tcs[0]['function']['arguments'])

@patch
def check(self:Chat, question, expected, grade_fn=_matches, llm_judge=False, judge=None, tag='answer', sp=_qa_sp):
    'Ask `question` stateless, extract the ```<tag> answer, and grade it against `expected`.'
    msgs = ([{'role': 'system', 'content': sp}] if sp else []) + [{'role': 'user', 'content': question}]
    a = extract_fence(resp_text(norm_resp(self.engine.create_chat_completion(msgs, **self._samp))), tag)
    ok = (judge or self).grades(question, expected, a) if (llm_judge or judge) else grade_fn(a, expected)
    return AttrDict(question=question, expected=expected, answer=a, ok=ok)

Chat.run_py, Chat.grades = core.Chat.run_py, core.Chat.grades

## AsyncChat

An async facade over `Chat` with the identical constructor (or pass an existing `Chat`): blocking llama.cpp calls run in a worker thread via `asyncio.to_thread`, so an event loop stays responsive. `await achat(msg)` for one turn; `async for c in await achat(msg, stream=True)` to stream. Everything else (`hist`, `use`, `print_hist`, callbacks, ...) is delegated to the wrapped `Chat`.

In [ ]:
#| export
class AsyncChat(GetAttr):
    "Async facade over `Chat`: same API; blocking llama.cpp calls run in a worker thread."
    _default = 'chat'
    def __init__(self, *args, **kwargs):
        self.chat = args[0] if args and isinstance(args[0], Chat) else Chat(*args, **kwargs)
    async def __call__(self, msg=None, stream=False, max_output_tokens=None,
                       cbs=None # extra callbacks for this turn only
    ):
        'Run one chat turn; `await` the result (an async chunk generator when `stream=True`).'
        if stream: return self._astream(msg, max_output_tokens, cbs)
        return await asyncio.to_thread(self.chat, msg, False, max_output_tokens, cbs)
    async def _astream(self, msg, max_output_tokens=None, cbs=None):
        g = self.chat(msg, stream=True, max_output_tokens=max_output_tokens, cbs=cbs)
        done = object()
        while (o := await asyncio.to_thread(next, g, done)) is not done: yield o
    def close(self): self.chat.close()
    async def __aenter__(self): return self
    async def __aexit__(self, *exc): self.close()

### Plumbing tests

`_FakeLlama` scripts responses so the whole pipeline - system prompt, think filtering, the tool loop with approval, streaming, usage - is tested without downloading a model.

In [ ]:
class _FakeLlama:
    "Scripted stand-in for `llama_cpp.Llama` for plumbing tests."
    def __init__(self, outs): self.outs, self.calls, self.n_tokens = list(outs), [], 42
    def n_ctx(self): return 2048
    def tokenize(self, b, add_bos=False, special=True): return list(b)
    def create_chat_completion(self, messages, stream=False, **kw):
        self.calls.append({'messages': messages, 'stream': stream, **kw})
        out = self.outs.pop(0)
        if not stream:
            return {'choices': [{'message': {'role': 'assistant', 'content': out}, 'finish_reason': 'stop'}],
                    'usage': {'prompt_tokens': 10, 'completion_tokens': 5, 'total_tokens': 15}}
        def gen():
            for i in range(0, len(out), 3):
                yield {'choices': [{'delta': {'content': out[i:i + 3]}, 'finish_reason': None}]}
            yield {'choices': [{'delta': {}, 'finish_reason': 'stop'}]}
        return gen()

In [ ]:
# basic turn: think split out, history recorded, usage tracked
fake = _FakeLlama(['<think>hmm</think>Hello!', 'Again!'])
chat = Chat(engine=fake, sp='SYS', think=False)
r = chat('hi')
test_eq(resp_text(r), 'Hello!')
test_eq(thought(r), 'hmm')
test_eq([m['role'] for m in chat.hist], ['user', 'assistant'])
test_eq((chat.use.total_tokens, chat.use.n, chat.token_count), (15, 1, 15))

# second turn re-sends history with thinking filtered out and the soft think switch in the system msg
chat('more')
sent = fake.calls[-1]['messages']
test_eq(sent[0], {'role': 'system', 'content': 'SYS\n/no_think'})
assert not any('<think>' in str(m.get('content')) for m in sent)
test_eq([m['role'] for m in sent], ['system', 'user', 'assistant', 'user'])

In [ ]:
# tool loop: <tool_call> tags parsed, tool executed via toolslm, result fed back, usage summed
fake = _FakeLlama(['<tool_call>{"name": "_add", "arguments": {"a": 2, "b": 3}}</tool_call>', 'The answer is 5.'])
chat = Chat(engine=fake, tools=[_add])
r = chat('add 2 and 3')
test_eq(resp_text(r), 'The answer is 5.')
roles = [m['role'] for m in chat.hist]
test_eq(roles, ['user', 'assistant', 'tool', 'assistant'])
test_eq(chat.hist[2]['content'], '5')
test_eq(chat.use.total_tokens, 30)
# the tool spec was passed to llama.cpp, and the follow-up request re-encodes arguments as JSON
test_eq(fake.calls[0]['tools'][0]['function']['name'], '_add')
tc_sent = first(m for m in fake.calls[1]['messages'] if m.get('tool_calls'))
test_eq(tc_sent['tool_calls'][0]['function']['arguments'], '{"a": 2, "b": 3}')
# ToolReminderCallback appended the reminder to the outgoing user message
assert 'system-reminder' in fake.calls[0]['messages'][0]['content']

In [ ]:
# denied tool call: approval consulted, denial recorded and fed back
fake = _FakeLlama(['<tool_call>{"name": "_add", "arguments": {"a": 1, "b": 1}}</tool_call>', 'I was not allowed.'])
chat = Chat(engine=fake, tools=[_add], approve=hitl_policy({'_add': 'dont_run'}))
r = chat('add')
test_eq(chat.hist[2]['content'], 'Denied by human operator')

# tool errors are reported back, not raised
fake = _FakeLlama(['<tool_call>{"name": "_add", "arguments": {"a": "x"}}</tool_call>', 'hmm.'])
chat = Chat(engine=fake, tools=[_add])
chat('add')
assert 'Error' in chat.hist[2]['content'] or 'error' in chat.hist[2]['content'].lower()

In [ ]:
# streaming: thinking renders as a blockquote, tool calls as ⏳ lines, history/usage finalised at the end
fake = _FakeLlama(['<think>let me add</think><tool_call>{"name": "_add", "arguments": {"a": 2, "b": 2}}</tool_call>',
                   'It is 4.'])
chat = Chat(engine=fake, tools=[_add])
out = ''.join(chat('add 2+2', stream=True))
assert '🧠 Thinking' in out and 'let me add' in out
assert '⏳' in out and '_add' in out and 'It is 4.' in out
test_eq(resp_text(chat.turn_res), 'It is 4.')
test_eq([m['role'] for m in chat.hist], ['user', 'assistant', 'tool', 'assistant'])
assert chat.use.n == 1 and chat.use.total_tokens > 0  # one turn: usage is summed across tool rounds

In [ ]:
# AsyncChat delegates to the same pipeline
from rishi.core import run_coro
fake = _FakeLlama(['ok', 'one two three'])
achat = AsyncChat(Chat(engine=fake))
test_eq(resp_text(run_coro(achat('hi'))), 'ok')
async def _collect():
    return ''.join([c async for c in await achat('count', stream=True)])
assert 'one two three' in run_coro(_collect())
test_eq(len(achat.hist), 4)

## Using Chat

A short tour, mirroring the litert one. These cells download and run a real GGUF model, so they are set to run manually rather than in the test suite.

In [ ]:
#| eval: false
chat = Chat(model_id=qwen3_06b, think=False, temp=0.0)
r = chat("Reply with exactly: pong")
assert 'pong' in resp_text(r).lower()
assert chat.hist[-1] is r and chat.hist[0]['role'] == 'user'
assert chat.use.total_tokens > 0 and chat.token_count > 0
print(chat.use); chat.print_hist()

### Tools and approval

Tool specs go both into the chat template (Qwen-style templates render them natively) and through our Hermes-tag parser, so tool calling works with plain GGUF chat models. `approve` and `hitl_policy` behave exactly as in the litert backend; a denied call is fed back as `Denied by human operator`.

In [ ]:
#| eval: false
def add(
    a: int, # first addend
    b: int  # second addend
) -> int:
    'Add two integers.'
    return a + b

chat = Chat(model_id=qwen3_06b, tools=[add], think=False, temp=0.0,
            sp='Use the tools to satisfy the request.')
r = chat('What is 12345 plus 67890? Use the add tool.')
assert '80235' in resp_text(r)
chat.print_hist()

In [ ]:
#| eval: false
chat = Chat(model_id=qwen3_06b, tools=[add], approve=hitl_policy({'add': 'dont_run'}),
            think=False, temp=0.0, sp='Use the tools to satisfy the request.')
r = chat('What is 2 plus 3? Use the add tool.')
assert any(m.get('role') == 'tool' and 'Denied' in str(m.get('content')) for m in chat.hist)

### Streaming and thinking

`stream=True` yields markdown chunks (thinking streams as a blockquote, tool calls as `⏳` lines) - render live with `display_stream`, exactly like the litert backend. With a Qwen3 model, `think=True` turns the thinking channel on.

In [ ]:
#| eval: false
ch = Chat(model_id=qwen3_06b, think=True)
for c in ch('Count: one two three', stream=True): print(c, end='', flush=True)
print('\n', ch.use)

In [ ]:
#| eval: false
display_stream(ch('And backwards?', stream=True))

### One-shot utilities

In [ ]:
#| eval: false
print(chat.classify('I absolutely loved this movie!', ['positive', 'negative']))

class Person:
    'Extract a person'
    def __init__(self,
        name: str, # their name
        age: int   # their age in years
    ): self.name, self.age = name, age

p = chat.structured('Alice is 30 years old.', Person)
print(p.name, p.age)

### AsyncChat

In [ ]:
#| eval: false
achat = AsyncChat(model_id=qwen3_06b, think=False, temp=0.0)
r = await achat('Reply with exactly: pong')
assert 'pong' in resp_text(r).lower()
async for c in await achat('Count to three.', stream=True): print(c, end='', flush=True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()